In [ ]:
# In[1]:

import pandas as pd
import pytz

# Timezone for UTC+8 as required
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into DataFrames (stored in kernel variables for later steps)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Ensure timestamp is integer type where possible
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = df['timestamp'].astype('Int64')  # allow NA if any

# Add timezone-aware datetime columns (UTC->Asia/Shanghai) for internal use
if 'timestamp' in metric_df.columns and not metric_df['timestamp'].isna().all():
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
if 'timestamp' in trace_df.columns and not trace_df['timestamp'].isna().all():
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
if 'timestamp' in log_df.columns and not log_df['timestamp'].isna().all():
    log_df['dt'] = pd.to_datetime(log_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
if 'timestamp' in error_logs_df.columns and not error_logs_df['timestamp'].isna().all():
    error_logs_df['dt'] = pd.to_datetime(error_logs_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# 1) For each file: rows, min timestamp, max timestamp, top 5 rows (sanity sample)
def file_summary(df, fname, sample_n=5):
    rows = int(df.shape[0])
    if rows > 0 and 'timestamp' in df.columns:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
    else:
        min_ts = None
        max_ts = None
    sample = df.head(sample_n)
    return {'file': fname, 'rows': rows, 'min_timestamp': min_ts, 'max_timestamp': max_ts, 'sample': sample}

metric_summary = file_summary(metric_df, 'metric.csv', sample_n=5)
trace_summary  = file_summary(trace_df,  'trace.csv',  sample_n=5)
log_summary    = file_summary(log_df,    'log.csv',    sample_n=5)
error_summary  = file_summary(error_logs_df, 'error_logs.csv', sample_n=5)

# Create compact combined summary table (without sample column)
combined_summary = pd.DataFrame([
    {'file': metric_summary['file'], 'rows': metric_summary['rows'], 'min_timestamp': metric_summary['min_timestamp'], 'max_timestamp': metric_summary['max_timestamp']},
    {'file': trace_summary['file'],  'rows': trace_summary['rows'],  'min_timestamp': trace_summary['min_timestamp'],  'max_timestamp': trace_summary['max_timestamp']},
    {'file': log_summary['file'],    'rows': log_summary['rows'],    'min_timestamp': log_summary['min_timestamp'],    'max_timestamp': log_summary['max_timestamp']},
    {'file': error_summary['file'],  'rows': error_summary['rows'],  'min_timestamp': error_summary['min_timestamp'],  'max_timestamp': error_summary['max_timestamp']},
])

# Extract samples separately (compact)
metric_sample = metric_summary['sample']
trace_sample  = trace_summary['sample']
log_sample    = log_summary['sample']
error_sample  = error_summary['sample']

# 2) Combined unique cmdb_id values and counts across all files (top 50)
cmdb_series_list = []
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'cmdb_id' in df.columns:
        cmdb_series_list.append(df['cmdb_id'].dropna().astype(str))
if cmdb_series_list:
    all_cmdbs = pd.concat(cmdb_series_list, ignore_index=True)
    cmdb_counts_df = all_cmdbs.value_counts().reset_index().rename(columns={'index':'cmdb_id', 0:'count'})
    cmdb_counts_df = cmdb_counts_df.head(50)
else:
    cmdb_counts_df = pd.DataFrame(columns=['cmdb_id','count'])

# 3) metric.csv: distinct kpi_name and counts (top 50)
if 'kpi_name' in metric_df.columns:
    metric_kpi_counts_df = metric_df['kpi_name'].dropna().astype(str).value_counts().reset_index().rename(columns={'index':'kpi_name', 0:'count'}).head(50)
else:
    metric_kpi_counts_df = pd.DataFrame(columns=['kpi_name','count'])

# 4) trace.csv: distinct trace_name and counts (top 50)
if 'trace_name' in trace_df.columns:
    trace_counts_df = trace_df['trace_name'].dropna().astype(str).value_counts().reset_index().rename(columns={'index':'trace_name', 0:'count'}).head(50)
else:
    trace_counts_df = pd.DataFrame(columns=['trace_name','count'])

# 5) log.csv: distinct log_name and counts (top 50)
if 'log_name' in log_df.columns:
    log_counts_df = log_df['log_name'].dropna().astype(str).value_counts().reset_index().rename(columns={'index':'log_name', 0:'count'}).head(50)
else:
    log_counts_df = pd.DataFrame(columns=['log_name','count'])

# 6) error_logs.csv: top 20 rows (timestamp, cmdb_id, message) and distinct cmdb_id counts in that file
if not error_logs_df.empty:
    cols = [c for c in ['timestamp','cmdb_id','message'] if c in error_logs_df.columns]
    error_top20_df = error_logs_df[cols].head(20)
    if 'cmdb_id' in error_logs_df.columns:
        error_cmdb_counts_df = error_logs_df['cmdb_id'].dropna().astype(str).value_counts().reset_index().rename(columns={'index':'cmdb_id', 0:'count'})
    else:
        error_cmdb_counts_df = pd.DataFrame(columns=['cmdb_id','count'])
else:
    error_top20_df = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    error_cmdb_counts_df = pd.DataFrame(columns=['cmdb_id','count'])

# Final compact outputs (variables to display)
combined_summary, metric_sample, trace_sample, log_sample, error_top20_df, cmdb_counts_df, metric_kpi_counts_df, trace_counts_df, log_counts_df, error_cmdb_counts_df

```
Out[1]:
```
```python
# Compose a concise plain-English summary based on earlier loaded DataFrames and summaries.
# The kernel already contains: combined_summary, metric_sample, trace_sample, log_sample,
# error_top20_df, cmdb_counts_df, metric_kpi_counts_df, trace_counts_df, log_counts_df, error_cmdb_counts_df

# Build human-readable summary
cs = combined_summary.set_index('file')
metric_rows = int(cs.loc['metric.csv','rows'])
metric_min_ts = int(cs.loc['metric.csv','min_timestamp'])
metric_max_ts = int(cs.loc['metric.csv','max_timestamp'])

trace_rows = int(cs.loc['trace.csv','rows'])
trace_min_ts = int(cs.loc['trace.csv','min_timestamp'])
trace_max_ts = int(cs.loc['trace.csv','max_timestamp'])

log_rows = int(cs.loc['log.csv','rows'])
log_min_ts = int(cs.loc['log.csv','min_timestamp'])
log_max_ts = int(cs.loc['log.csv','max_timestamp'])

error_rows = int(cs.loc['error_logs.csv','rows'])
error_min_ts = cs.loc['error_logs.csv','min_timestamp']
error_max_ts = cs.loc['error_logs.csv','max_timestamp']

# Top cmdb_id counts (up to 10 shown)
top_cmdb = cmdb_counts_df.head(10).copy()
top_cmdb_list = [f"{r['cmdb_id']}:{int(r['count'])}" for _,r in top_cmdb.iterrows()]
top_cmdb_str = ", ".join(top_cmdb_list) if top_cmdb_list else "None"

# Metric kpi top 5
top_kpis = metric_kpi_counts_df.head(5).copy()
top_kpis_list = [f"{r['kpi_name']}({int(r['count'])})" for _,r in top_kpis.iterrows()]
top_kpis_str = ", ".join(top_kpis_list) if top_kpis_list else "None"
metric_kpi_distinct = int(metric_kpi_counts_df.shape[0])

# Trace top 5
top_traces = trace_counts_df.head(5).copy()
top_traces_list = [f"{r['trace_name']}({int(r['count'])})" for _,r in top_traces.iterrows()]
top_traces_str = ", ".join(top_traces_list) if top_traces_list else "None"
trace_name_distinct = int(trace_counts_df.shape[0])

# Log top names
top_logs = log_counts_df.head(5).copy()
top_logs_list = [f"{r['log_name']}({int(r['count'])})" for _,r in top_logs.iterrows()]
top_logs_str = ", ".join(top_logs_list) if top_logs_list else "None"
log_name_distinct = int(log_counts_df.shape[0])

# Error logs status
error_present = "no rows (empty file)" if error_rows == 0 else f"{error_rows} rows"
error_top_sample = error_top20_df.head(5) if not error_top20_df.empty else None
error_cmdb_counts = error_cmdb_counts_df.shape[0]

summary = (
    "Telemetry load summary (plain English):\n\n"
    f"1) Files:\n"
    f" - metric.csv: {metric_rows} rows, min_timestamp={metric_min_ts}, max_timestamp={metric_max_ts}.\n"
    f" - trace.csv:  {trace_rows} rows, min_timestamp={trace_min_ts}, max_timestamp={trace_max_ts}.\n"
    f" - log.csv:    {log_rows} rows, min_timestamp={log_min_ts}, max_timestamp={log_max_ts}.\n"
    f" - error_logs.csv: {error_present}.\n\n"
    f"2) Combined cmdb_id counts (top up to 10 shown): {top_cmdb_str}\n\n"
    f"3) metric.csv: {metric_kpi_distinct} distinct kpi_name values. Top 5 by row count: {top_kpis_str}\n\n"
    f"4) trace.csv: {trace_name_distinct} distinct trace_name values. Top 5 by row count: {top_traces_str}\n\n"
    f"5) log.csv: {log_name_distinct} distinct log_name values. Top names: {top_logs_str}\n\n"
    f"6) error_logs.csv: {error_present}. Distinct cmdb_id in error_logs: {error_cmdb_counts}.\n\n"
    "Notes: samples for each file were loaded (not fully printed here). If you want the top-5 sample rows again, ask for metric/trace/log samples or error top20."
)

# Display the summary string (as the execution result)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows  min_timestamp  max_timestamp
0      metric.csv  29565   1.616636e+09   1.616638e+09
1       trace.csv   7077   1.616636e+09   1.616638e+09
2         log.csv    483   1.616636e+09   1.616638e+09
3  error_logs.csv      0            NaN            NaN,     timestamp cmdb_id                                     kpi_name         value                        dt
0  1616635800    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09 2021-03-25 09:30:00+08:00
1  1616635800    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.730820e+01 2021-03-25 09:30:00+08:00
2  1616635800    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  2.082236e+08 2021-03-25 09:30:00+08:00
3  1616635800    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.846119e+07 2021-03-25 09:30:00+08:00
4  1616635800    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad  5.574000e-01 2021-03-25 09:30:00+08:00,     timestamp cmdb_id                       trace_name        value                        dt
0  1616635800    IG01         trace.self.duration_mean     0.343743 2021-03-25 09:30:00+08:00
1  1616635800    IG01          trace.self.duration_p95     1.229150 2021-03-25 09:30:00+08:00
2  1616635800    IG01             trace.self.row_count  2032.000000 2021-03-25 09:30:00+08:00
3  1616635800    IG01  trace.to_Tomcat01.duration_mean     0.378241 2021-03-25 09:30:00+08:00
4  1616635800    IG01   trace.to_Tomcat01.duration_p95     1.182700 2021-03-25 09:30:00+08:00,     timestamp   cmdb_id          log_name   value                        dt
0  1616635800  Tomcat01   log.error_count    0.00 2021-03-25 09:30:00+08:00
1  1616635800  Tomcat01  log.gc_pause_max    0.08 2021-03-25 09:30:00+08:00
2  1616635800  Tomcat01     log.row_count  544.00 2021-03-25 09:30:00+08:00
3  1616635800  Tomcat02   log.error_count    0.00 2021-03-25 09:30:00+08:00
4  1616635800  Tomcat02     log.row_count  516.00 2021-03-25 09:30:00+08:00, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],          cmdb_id  count
0        Mysql02   4788
1        Mysql01   4760
2           MG01   2846
3           MG02   2815
4       Tomcat04   2624
..           ...    ...
24  ServiceTest9    124
25  ServiceTest5    124
26  ServiceTest4    124
27  ServiceTest6    124
28  ServiceTest7    124

[29 rows x 2 columns],                                              kpi_name  count
0                  OSLinux-OSLinux_ZABBIX_Host_Uptime    347
1       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    346
2          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    346
3        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    346
4      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    345
..                                                ...    ...
45     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    139
46     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    139
47                    OSLinux-CPU_CPU-0_SingleCpuUtil    137
48         OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity    134
49  OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPer...    133

[50 rows x 2 columns],                            trace_name  count
0            trace.self.duration_mean    372
1             trace.self.duration_p95    372
2                trace.self.row_count    372
3             trace.to_MG01.row_count    247
4          trace.to_MG01.duration_p95    247
..                                ...    ...
45      trace.from_dockerB2.row_count     62
46   trace.from_dockerB2.duration_p95     62
47  trace.from_dockerB2.duration_mean     62
48      trace.from_dockerA2.row_count     62
49  trace.from_dockerB1.duration_mean     62

[50 rows x 2 columns],            log_name  count
0   log.error_count    186
1     log.row_count    186
2  log.gc_pause_max    111, Empty DataFrame
Columns: [cmdb_id, count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse in-memory DataFrames: metric_df, trace_df, log_df (assumed loaded previously)
# Ensure 'value' columns are numeric for quantile calculations
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value']  = pd.to_numeric(trace_df['value'],  errors='coerce')
log_df['value']    = pd.to_numeric(log_df['value'],    errors='coerce')

# 1) metric.csv: group by (cmdb_id, kpi_name) -> compute P95 and count
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), count='count')
    .reset_index()
)

# 2) trace.csv: group by (cmdb_id, trace_name)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), count='count')
    .reset_index()
)

# 3) log.csv: group by (cmdb_id, log_name)
log_thresholds = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), count='count')
    .reset_index()
)

# For compact outputs: counts of distinct series and top 20 rows sorted by p95 desc
metric_count = metric_thresholds.shape[0]
metric_top20 = metric_thresholds.sort_values('p95', ascending=False).head(20)[['cmdb_id','kpi_name','p95','count']]

trace_count = trace_thresholds.shape[0]
trace_top20 = trace_thresholds.sort_values('p95', ascending=False).head(20)[['cmdb_id','trace_name','p95','count']]

log_count = log_thresholds.shape[0]
log_top20 = log_thresholds.sort_values('p95', ascending=False).head(20)[['cmdb_id','log_name','p95','count']]

# Display the compact summaries (variables returned)
metric_count, metric_top20, trace_count, trace_top20, log_count, log_top20

```
Out[2]:
```
Summary of global P95 thresholds (computed on the full unfiltered series):

1) metric.csv
- Number of distinct (cmdb_id, kpi_name) series: 1,754.
- Top series by P95 (examples from the top 20):
  - Mysql02 · Mysql Innodb "data written": p95 ≈ 2.827e+12 (count=30)
  - Mysql02 · Innodb "buffer pool read requests": p95 ≈ 1.666e+12 (count=30)
  - Mysql02 · Innodb "os log written": p95 ≈ 7.434e+11 (count=30)
  - Mysql01 · Innodb "data written": p95 ≈ 6.377e+11 (count=30)
  - Mysql02 · "Rows Read": p95 ≈ 5.445e+11 (count=30)
  - Redis01 · Redis metrics (e.g., total_commands_processed or keyspace_hits): p95 on the order of 1e+9 (count=20)
  - Several Tomcat JVM memory KPIs appear with p95 ≈ 4.19e+09 (count=6)
- Notes: top metric p95 values are dominated by Mysql and Redis throughput/byte counters (large magnitudes). Most top metric groups have counts like 30 or 20 (the group sample sizes vary by series).

2) trace.csv
- Number of distinct (cmdb_id, trace_name) series: 249.
- Top series by P95 (top examples):
  - dockerB1 · trace.self.row_count: p95 = 8811.5 (count=31)
  - Tomcat01 · trace.self.row_count: p95 = 4860.5 (count=31)
  - Tomcat03 · trace.self.row_count: p95 = 4843.5 (count=31)
  - Tomcat02 · trace.self.row_count: p95 = 4807.5 (count=31)
  - Tomcat04 · trace.self.row_count: p95 = 4718.5 (count=31)
  - Additional high p95 entries are various trace row_count or from/to series (many with count=31).
- Notes: trace top p95 values are mainly row_count measures from application components (Docker/Tomcat/IG/MG).

3) log.csv
- Number of distinct (cmdb_id, log_name) series: 16.
- Top series by P95:
  - apache01 · log.row_count: p95 = 1416.5 (count=31)
  - apache02 · log.row_count: p95 = 1404.0 (count=31)
  - Tomcat04 · log.row_count: p95 = 717.5 (count=31)
  - Tomcat03 · log.row_count: p95 = 717.0 (count=31)
  - Tomcat01 · log.row_count: p95 = 715.5 (count=31)
  - Tomcat02 · log.row_count: p95 = 711.0 (count=31)
  - Tomcat log.gc_pause_max values: p95 around 0.08–0.10 (counts vary ~26–29)
  - log.error_count p95 is 0.0 for several components (count=31).
- Notes: log series are few; top p95s are log row counts for web servers and Tomcats; GC pause p95s are small numeric values.

Additional notes:
- P95 values were computed globally on the full unfiltered value series for each (component, KPI) group.
- The full threshold tables have been stored in the kernel variables metric_thresholds, trace_thresholds, and log_thresholds for subsequent steps.

The original code execution output of IPython Kernel is also provided below for reference:

(1754,        cmdb_id                                           kpi_name           p95  count
669    Mysql02               Mysql-MySQL_3306_Innodb data written  2.827178e+12     30
658    Mysql02  Mysql-MySQL_3306_Innodb buffer pool read requests  1.665615e+12     30
679    Mysql02             Mysql-MySQL_3306_Innodb os log written  7.434022e+11     30
444    Mysql01               Mysql-MySQL_3306_Innodb data written  6.376661e+11     30
709    Mysql02                         Mysql-MySQL_3306_Rows Read  5.444835e+11     30
640    Mysql02                 Mysql-MySQL_3306_Handler Read Prev  5.442917e+11     30
617    Mysql02                        Mysql-MySQL_3306_Bytes Sent  1.905364e+11     30
392    Mysql01                        Mysql-MySQL_3306_Bytes Sent  7.947066e+10     30
666    Mysql02                  Mysql-MySQL_3306_Innodb data read  3.317295e+10     30
454    Mysql01             Mysql-MySQL_3306_Innodb os log written  9.331333e+09     30
936    Redis01  redis-Redis_6379_Redis  (total_commands_proces...  7.040233e+09     20
1046   Redis02  redis-Redis_6379_Redis  (total_commands_proces...  6.193078e+09     20
924    Redis01            redis-Redis_6379_Redis  (keyspace_hits)  5.394503e+09     20
1034   Redis02            redis-Redis_6379_Redis  (keyspace_hits)  4.573101e+09     20
929    Redis01                redis-Redis_6379_Redis  (maxmemory)  4.294967e+09     20
1039   Redis02                redis-Redis_6379_Redis  (maxmemory)  4.294967e+09     20
1317  Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
1319  Tomcat02           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
1438  Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
1559  Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6, 249,       cmdb_id                     trace_name     p95  count
224  dockerB1           trace.self.row_count  8811.5     31
116  Tomcat01           trace.self.row_count  4860.5     31
152  Tomcat03           trace.self.row_count  4843.5     31
134  Tomcat02           trace.self.row_count  4807.5     31
170  Tomcat04           trace.self.row_count  4718.5     31
188  dockerA1           trace.self.row_count  4200.5     31
206  dockerA2           trace.self.row_count  4187.0     31
242  dockerB2           trace.self.row_count  3045.5     31
17       IG02           trace.self.row_count  2859.0     31
2        IG01           trace.self.row_count  2833.5     31
95       MG02           trace.self.row_count  2161.5     31
56       MG01           trace.self.row_count  2138.5     31
218  dockerB1      trace.from_MG02.row_count   671.5     31
104      MG02    trace.to_dockerB1.row_count   671.5     31
65       MG01    trace.to_dockerB1.row_count   644.5     31
215  dockerB1      trace.from_MG01.row_count   644.5     31
227  dockerB1        trace.to_MG01.row_count   584.5     31
50       MG01  trace.from_dockerB1.row_count   584.5     31
230  dockerB1        trace.to_MG02.row_count   577.5     31
89       MG02  trace.from_dockerB1.row_count   577.5     31, 16,      cmdb_id          log_name        p95  count
13  apache01     log.row_count  1416.5000     31
15  apache02     log.row_count  1404.0000     31
11  Tomcat04     log.row_count   717.5000     31
8   Tomcat03     log.row_count   717.0000     31
2   Tomcat01     log.row_count   715.5000     31
5   Tomcat02     log.row_count   711.0000     31
4   Tomcat02  log.gc_pause_max     0.1000     26
10  Tomcat04  log.gc_pause_max     0.0900     29
1   Tomcat01  log.gc_pause_max     0.0865     28
7   Tomcat03  log.gc_pause_max     0.0800     28
0   Tomcat01   log.error_count     0.0000     31
3   Tomcat02   log.error_count     0.0000     31
6   Tomcat03   log.error_count     0.0000     31
9   Tomcat04   log.error_count     0.0000     31
12  apache01   log.error_count     0.0000     31
14  apache02   log.error_count     0.0000     31)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames and thresholds in kernel:
# metric_df, trace_df, log_df, metric_thresholds, trace_thresholds, log_thresholds

# Ensure dt tz-aware columns exist (UTC->Asia/Shanghai)
for df in (metric_df, trace_df, log_df):
    if 'dt' not in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
    else:
        # Ensure tz-aware (in case)
        if df['dt'].dt.tz is None:
            df['dt'] = df['dt'].dt.tz_localize(pytz.utc).dt.tz_convert(tz)

# Define incident window: 2021-03-25 09:30 to 10:00 Asia/Shanghai (UTC+8).
start_dt = tz.localize(pd.Timestamp('2021-03-25 09:30:00'))
end_dt   = tz.localize(pd.Timestamp('2021-03-25 10:00:00'))  # exclusive

# Helper to process one dataset
def find_anomalies(window_df, thresholds_df, key_name, value_col='value'):
    # Merge on cmdb_id and KPI/trace/log name (thresholds already computed globally)
    merged = window_df.merge(thresholds_df, how='left', left_on=['cmdb_id', key_name], right_on=['cmdb_id', key_name], suffixes=('','_thr'))
    # Drop rows without threshold (cannot decide anomaly)
    merged = merged[~merged['p95'].isna()].copy()
    # Ensure numeric value
    merged[value_col] = pd.to_numeric(merged[value_col], errors='coerce')
    # Flag anomaly where value >= p95
    merged['is_anom'] = merged[value_col] >= merged['p95']
    # Keep only anomaly rows
    anoms = merged[merged['is_anom']].copy()
    return merged, anoms

# Filter window for each dataset
metric_window = metric_df[(metric_df['dt'] >= start_dt) & (metric_df['dt'] < end_dt)].copy()
trace_window  = trace_df[(trace_df['dt']  >= start_dt) & (trace_df['dt']  < end_dt)].copy()
log_window    = log_df[(log_df['dt']    >= start_dt) & (log_df['dt']    < end_dt)].copy()

# Find anomalies by merging with thresholds
_, metric_anoms_window = find_anomalies(metric_window, metric_thresholds, 'kpi_name', value_col='value')
_, trace_anoms_window  = find_anomalies(trace_window,  trace_thresholds,  'trace_name', value_col='value')
_, log_anoms_window    = find_anomalies(log_window,    log_thresholds,    'log_name', value_col='value')

# Store anomaly tables in kernel variables (per instruction)
# metric_anoms_window, trace_anoms_window, log_anoms_window already assigned

# Aggregate helpers
def aggregate_series_anoms(anoms_df, name_col, value_col='value', top_n=20):
    if anoms_df.empty:
        total_anom_rows = 0
        distinct_series = 0
        top_series = pd.DataFrame(columns=[ 'cmdb_id', name_col, 'anomaly_count', 'earliest_anomaly_timestamp','max_value','p95','max_breach_ratio'])
        top_components = pd.DataFrame(columns=['cmdb_id','total_anomaly_count','distinct_faulty_kpis'])
        return total_anom_rows, distinct_series, top_series, top_components

    total_anom_rows = int(anoms_df.shape[0])
    distinct_series = int(anoms_df.drop_duplicates(subset=['cmdb_id', name_col]).shape[0])

    # Series-level aggregation
    grp = anoms_df.groupby(['cmdb_id', name_col], as_index=False)
    agg = grp.agg(
        anomaly_count = ('is_anom', 'sum'),
        earliest_anomaly_timestamp = ('timestamp', 'min'),
        max_value = (value_col, 'max'),
        p95 = ('p95', 'first')
    )
    # Compute max_breach_ratio safely
    agg['max_breach_ratio'] = np.where(agg['p95'] > 0, agg['max_value'] / agg['p95'], np.inf)
    # Convert earliest timestamp to int (already int-like)
    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].astype('Int64')
    top_series = agg.sort_values('anomaly_count', ascending=False).head(top_n).reset_index(drop=True)

    # Component-level aggregation: total anomalies per cmdb_id and distinct faulty KPIs
    comp_grp = anoms_df.groupby('cmdb_id', as_index=False)
    comp_agg = comp_grp.agg(
        total_anomaly_count = ('is_anom', 'sum'),
        distinct_faulty_kpis = (name_col, lambda s: s.nunique())
    )
    top_components = comp_agg.sort_values('total_anomaly_count', ascending=False).head(10).reset_index(drop=True)

    return int(total_anom_rows), int(distinct_series), top_series, top_components

# Compute aggregates for metric, trace, log
metric_total_anom_rows, metric_distinct_series_with_anoms, metric_top20, metric_comp_top10 = aggregate_series_anoms(metric_anoms_window, 'kpi_name', value_col='value', top_n=20)
trace_total_anom_rows, trace_distinct_series_with_anoms, trace_top20, trace_comp_top10    = aggregate_series_anoms(trace_anoms_window,  'trace_name', value_col='value', top_n=20)
log_total_anom_rows, log_distinct_series_with_anoms, log_top20, log_comp_top10            = aggregate_series_anoms(log_anoms_window,    'log_name', value_col='value', top_n=20)

# Keep final displayed outputs compact as requested
metric_total_anom_rows, metric_distinct_series_with_anoms, metric_top20, metric_comp_top10, \
trace_total_anom_rows, trace_distinct_series_with_anoms, trace_top20, trace_comp_top10, \
log_total_anom_rows, log_distinct_series_with_anoms, log_top20, log_comp_top10

```
Out[3]:
```
Here is a concise plain-English summary of the anomaly-detection results for the incident window (2021-03-25 09:30–10:00 Asia/Shanghai) based on the joined P95 thresholds.

Metric anomalies
- Total anomaly rows: 15,795
- Distinct (cmdb_id, kpi_name) series with anomalies: 1,684
- Top series (by anomaly_count) — showing cmdb_id, kpi_name, anomaly_count, earliest_anomaly_timestamp (epoch sec), max_value, p95, max_breach_ratio:
  1. IG01 · JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed — 30, 1616635800, max=58,461,192.0, p95=58,461,192.0, ratio=1.0
  2. Tomcat03 · (many Tomcat session/log KPIs with p95=0) — 30, 1616635800, max=0.0, p95=0.0, ratio=inf
  3. Tomcat03 · Tomcat-Threads...MaxThreads — 30, 1616635800, max=500.0, p95=500.0, ratio=1.0
  4. Tomcat03 · SessionActiveCounter — 30, 1616635800, max=0.0, p95=0.0, ratio=inf
  5. Tomcat03 · several other session/log KPIs (p95=0) — 30, 1616635800, max=0.0, p95=0.0, ratio=inf
  (Top-20 were returned; many top metric groups are Tomcat series and large-volume DB counters.)
- Top affected components by total anomaly count (top 10):
  1. Mysql01 — total_anomaly_count=2,531 (distinct faulty KPIs=224)
  2. Mysql02 — 2,411 (224)
  3. Tomcat03 — 1,235 (115)
  4. Tomcat02 — 1,187 (118)
  5. Tomcat04 — 1,177 (112)
  6. Tomcat01 — 1,102 (109)
  7. Redis02 — 1,002 (98)
  8. Redis01 — 975 (105)
  9. MG02 — 748 (99)
 10. MG01 — 725 (97)

Trace anomalies
- Total anomaly rows: 483
- Distinct (cmdb_id, trace_name) series with anomalies: 249
- Top trace series (top examples by anomaly_count):
  1. Tomcat03 · trace.self.duration_p95 — anomaly_count=15, earliest=1616635920, max=1.005, p95=1.004, ratio≈1.000996
  2. dockerB1 · trace.self.duration_p95 — 9, 1616636160, max=1.006, p95=1.006, ratio=1.0
  3. Tomcat04 · trace.self.duration_p95 — 8, 1616636460, max=1.0152, p95=1.005, ratio≈1.01015
  4. Tomcat02 · trace.self.duration_p95 — 5, 1616636160, max=1.0256, p95=1.005, ratio≈1.0205
  5. dockerB2 · trace.self.duration_p95 — 4, 1616636640, max=1.008, p95=1.008, ratio=1.0
  - IG01 and IG02 show several series with breach ratios >1.2 (e.g., IG01 trace.self.duration and row_count).
- Top affected components by total anomaly count (top 10):
  1. MG02 — 75 anomalies (39 distinct faulty traces)
  2. MG01 — 74 (39)
  3. Tomcat03 — 43 (18)
  4. dockerB1 — 40 (18)
  5. Tomcat04 — 38 (18)
  6. Tomcat02 — 36 (18)
  7. dockerB2 — 35 (18)
  8. dockerA1 — 33 (18)
  9. IG01 — 30 (15)
 10. dockerA2 — 28 (18)

Log anomalies
- Total anomaly rows: 212
- Distinct (cmdb_id, log_name) series with anomalies: 16
- Top log series (top examples):
  1. Tomcat01 · log.error_count — anomaly_count=30, earliest=1616635800, max=0.00, p95=0.0, ratio=inf
  2. Tomcat02 · log.error_count — 30, 1616635800, max=0.00, p95=0.0, ratio=inf
  3. Tomcat03 · log.error_count — 30, ...
  4. Tomcat04 · log.error_count — 30, ...
  5. apache01/apache02 · log.error_count — 30 each, ...
  6. Tomcat03 · log.gc_pause_max — 11, earliest=1616635860, max=0.08, p95=0.08, ratio≈1.0
  7. Tomcat02/01/04 show gc_pause_max or row_count anomalies with breach ratios up to ≈1.27
- Top affected components by total anomaly count (top 10):
  1. Tomcat03 — 43 anomalies (3 distinct faulty log_names)
  2. Tomcat04 — 36 (3)
  3. Tomcat02 — 35 (3)
  4. Tomcat01 — 34 (3)
  5. apache01 — 32 (2)
  6. apache02 — 32 (2)

Additional notes
- Earliest anomaly timestamps in many series equal 1616635800 (2021-03-25 09:30:00 +08), meaning many anomalies occur right at the window start.
- Several KPI/ log series have p95 == 0, producing infinite breach ratio when max_value == 0 (these appear as ratio=inf in results). These need careful interpretation (zero baseline).
- Joined, window-filtered anomaly tables are stored in kernel variables metric_anoms_window, trace_anoms_window, log_anoms_window for follow-up analysis.

The original code execution output of IPython Kernel is also provided below for reference:

(15795, 1684,      cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_timestamp   max_value         p95  max_breach_ratio
0       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             30                  1616635800  58461192.0  58461192.0               1.0
1   Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             30                  1616635800         0.0         0.0               inf
2   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             30                  1616635800       500.0       500.0               1.0
3   Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter             30                  1616635800         0.0         0.0               inf
4   Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             30                  1616635800         0.0         0.0               inf
5   Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions             30                  1616635800         0.0         0.0               inf
6   Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             30                  1616635800         0.0         0.0               inf
7   Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             30                  1616635800         0.0         0.0               inf
8   Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             30                  1616635800         0.0         0.0               inf
9   Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             30                  1616635800         0.0         0.0               inf
10  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             30                  1616635800       100.0       100.0               1.0
11  Tomcat02     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             30                  1616635800         0.0         0.0               inf
12  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             30                  1616635800       100.0       100.0               1.0
13  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             30                  1616635800         0.0         0.0               inf
14  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             30                  1616635800         0.0         0.0               inf
15  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             30                  1616635800         0.0         0.0               inf
16  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             30                  1616635800         0.0         0.0               inf
17  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             30                  1616635800       994.0       994.0               1.0
18  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             30                  1616635800         0.0         0.0               inf
19  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             30                  1616635800      1093.0      1093.0               1.0,     cmdb_id  total_anomaly_count  distinct_faulty_kpis
0   Mysql01                 2531                   224
1   Mysql02                 2411                   224
2  Tomcat03                 1235                   115
3  Tomcat02                 1187                   118
4  Tomcat04                 1177                   112
5  Tomcat01                 1102                   109
6   Redis02                 1002                    98
7   Redis01                  975                   105
8      MG02                  748                    99
9      MG01                  725                    97, 483, 249,      cmdb_id                        trace_name  anomaly_count  earliest_anomaly_timestamp    max_value          p95  max_breach_ratio
0   Tomcat03           trace.self.duration_p95             15                  1616635920     1.005000     1.004000          1.000996
1   dockerB1           trace.self.duration_p95              9                  1616636160     1.006000     1.006000          1.000000
2   Tomcat04           trace.self.duration_p95              8                  1616636460     1.015200     1.005000          1.010149
3   Tomcat02           trace.self.duration_p95              5                  1616636160     1.025600     1.005000          1.020498
4   dockerB2           trace.self.duration_p95              4                  1616636640     1.008000     1.008000          1.000000
5   dockerB2        trace.to_MG02.duration_p95              3                  1616636640     1.015500     1.015300          1.000197
6       MG02  trace.from_dockerB2.duration_p95              3                  1616636640     1.015500     1.015300          1.000197
7       IG01           trace.self.duration_p95              2                  1616636160     4.091550     3.203275          1.277302
8       IG01              trace.self.row_count              2                  1616636640  3592.000000  2833.500000          1.267690
9       IG01   trace.to_Tomcat03.duration_mean              2                  1616636940     0.810517     0.766809          1.057001
10      IG01    trace.to_Tomcat03.duration_p95              2                  1616636640     4.127150     3.638875          1.134183
11      IG01       trace.to_Tomcat03.row_count              2                  1616637300   446.000000   366.000000          1.218579
12      IG01   trace.to_Tomcat04.duration_mean              2                  1616637000     0.828082     0.799871          1.035269
13      IG01    trace.to_Tomcat04.duration_p95              2                  1616636640     4.093650     3.161700          1.294762
14      IG01       trace.to_Tomcat04.row_count              2                  1616637300   439.000000   365.000000          1.202740
15      IG02       trace.to_Tomcat01.row_count              2                  1616636640   432.000000   349.000000          1.237822
16      IG02              trace.self.row_count              2                  1616636640  3446.000000  2859.000000          1.205317
17      IG02       trace.to_Tomcat02.row_count              2                  1616636640   425.000000   379.000000          1.121372
18      IG02   trace.to_Tomcat02.duration_mean              2                  1616636160     0.894682     0.768069          1.164845
19      IG01   trace.to_Tomcat01.duration_mean              2                  1616636160     0.791096     0.744220          1.062987,     cmdb_id  total_anomaly_count  distinct_faulty_kpis
0      MG02                   75                    39
1      MG01                   74                    39
2  Tomcat03                   43                    18
3  dockerB1                   40                    18
4  Tomcat04                   38                    18
5  Tomcat02                   36                    18
6  dockerB2                   35                    18
7  dockerA1                   33                    18
8      IG01                   30                    15
9  dockerA2                   28                    18, 212, 16,      cmdb_id          log_name  anomaly_count  earliest_anomaly_timestamp  max_value        p95  max_breach_ratio
0   Tomcat01   log.error_count             30                  1616635800       0.00     0.0000               inf
1   Tomcat02   log.error_count             30                  1616635800       0.00     0.0000               inf
2   Tomcat03   log.error_count             30                  1616635800       0.00     0.0000               inf
3   Tomcat04   log.error_count             30                  1616635800       0.00     0.0000               inf
4   apache01   log.error_count             30                  1616635800       0.00     0.0000               inf
5   apache02   log.error_count             30                  1616635800       0.00     0.0000               inf
6   Tomcat03  log.gc_pause_max             11                  1616635860       0.08     0.0800          1.000000
7   Tomcat04  log.gc_pause_max              4                  1616635860       0.09     0.0900          1.000000
8   Tomcat02  log.gc_pause_max              3                  1616636760       0.11     0.1000          1.100000
9   Tomcat02     log.row_count              2                  1616636640     891.00   711.0000          1.253165
10  Tomcat01  log.gc_pause_max              2                  1616636160       0.11     0.0865          1.271676
11  Tomcat01     log.row_count              2                  1616636640     874.00   715.5000          1.221523
12  Tomcat04     log.row_count              2                  1616637300     866.00   717.5000          1.206969
13  Tomcat03     log.row_count              2                  1616636640     890.00   717.0000          1.241283
14  apache01     log.row_count              2                  1616636640    1770.00  1416.5000          1.249559
15  apache02     log.row_count              2                  1616636640    1749.00  1404.0000          1.245726,     cmdb_id  total_anomaly_count  distinct_faulty_kpis
0  Tomcat03                   43                     3
1  Tomcat04                   36                     3
2  Tomcat02                   35                     3
3  Tomcat01                   34                     3
4  apache01                   32                     2
5  apache02                   32                     2)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_anoms_window from kernel
# Define top-10 components (as specified)
top_components = ["Mysql01","Mysql02","Tomcat03","Tomcat02","Tomcat04","Tomcat01","Redis02","Redis01","MG02","MG01"]

# Filter anomaly rows to only these components
df = metric_anoms_window[metric_anoms_window['cmdb_id'].isin(top_components)].copy()

# Ensure required columns exist and proper dtypes
for col in ['timestamp','cmdb_id','kpi_name','value','p95']:
    if col not in df.columns:
        raise KeyError(f"Required column missing: {col}")
df['timestamp'] = df['timestamp'].astype('int64')
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df['p95'] = pd.to_numeric(df['p95'], errors='coerce')

# Sort for run detection
df = df.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

# Detect runs where adjacent timestamps differ by exactly 60 seconds
df['prev_ts'] = df.groupby(['cmdb_id','kpi_name'])['timestamp'].shift(1)
df['gap'] = (df['timestamp'] - df['prev_ts']) != 60
df['gap'] = df['gap'] | df['prev_ts'].isna()
# run sequence id within each (cmdb_id,kpi_name)
df['run_seq'] = df.groupby(['cmdb_id','kpi_name'])['gap'].cumsum()

# Aggregate consecutive runs into fault records
grp = df.groupby(['cmdb_id','kpi_name','run_seq'], as_index=False)
faults = grp.agg(
    fault_start_timestamp = ('timestamp', 'min'),
    fault_end_timestamp   = ('timestamp', 'max'),
    point_count           = ('timestamp', 'count'),
    max_value             = ('value', 'max'),
    p95                   = ('p95', 'first')
)

# duration_minutes equals point_count per spec
faults['duration_minutes'] = faults['point_count']

# earliest_anomaly_timestamp same as start
faults['earliest_anomaly_timestamp'] = faults['fault_start_timestamp']

# compute max_breach_ratio with p95==0 -> inf
faults['max_breach_ratio'] = np.where(faults['p95'] == 0,
                                      np.inf,
                                      faults['max_value'] / faults['p95'])

# severity_score = max_breach_ratio * sqrt(duration_minutes)
faults['severity_score'] = faults['max_breach_ratio'] * np.sqrt(faults['duration_minutes'])

# Filter out runs with point_count < 3
metric_fault_runs_top = faults[faults['point_count'] >= 3].copy()

# Prepare compact outputs
metric_fault_run_count = int(metric_fault_runs_top.shape[0])  # total number of fault runs found

if metric_fault_run_count > 0:
    earliest_fault_start_ts = int(metric_fault_runs_top['fault_start_timestamp'].min())
else:
    earliest_fault_start_ts = None

# Top 50 fault records sorted by severity_score descending
metric_fault_runs_top = metric_fault_runs_top.sort_values('severity_score', ascending=False).reset_index(drop=True)
metric_fault_runs_top_display = metric_fault_runs_top.head(50)[[
    'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
    'point_count','duration_minutes','max_value','p95','max_breach_ratio',
    'earliest_anomaly_timestamp','severity_score'
]]

# Return compact outputs (variables shown)
metric_fault_run_count, earliest_fault_start_ts, metric_fault_runs_top_display

```
Out[4]:
```
```python
# Summarize the fault-segmentation results (plain English) as a single string.
import pandas as pd
import pytz

# Reuse kernel variables: metric_fault_run_count, earliest_fault_start_ts, metric_fault_runs_top_display
# Defensive fetch (assume variables exist in kernel as per prior steps)
count = metric_fault_run_count if 'metric_fault_run_count' in globals() else None
earliest_ts = earliest_fault_start_ts if 'earliest_fault_start_ts' in globals() else None
top_df = metric_fault_runs_top_display if 'metric_fault_runs_top_display' in globals() else pd.DataFrame()

tz = pytz.timezone('Asia/Shanghai')
if earliest_ts is not None:
    earliest_dt = pd.to_datetime(int(earliest_ts), unit='s', utc=True).tz_convert(tz)
    earliest_dt_str = earliest_dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    earliest_dt_str = "N/A"

# Prepare up to 5 example fault records for concise illustration
examples = []
if not top_df.empty:
    for _, r in top_df.head(5).iterrows():
        kpi_short = (r['kpi_name'][:80] + '...') if len(r['kpi_name'])>80 else r['kpi_name']
        p95_val = float(r['p95']) if pd.notna(r['p95']) else None
        max_val = float(r['max_value']) if pd.notna(r['max_value']) else None
        ratio = r['max_breach_ratio']
        # Format numeric fields compactly
        examples.append(
            f"- {r['cmdb_id']} · {kpi_short} | start={int(r['fault_start_timestamp'])}, "
            f"end={int(r['fault_end_timestamp'])}, points={int(r['point_count'])}, "
            f"max={max_val}, p95={p95_val}, severity={float(r['severity_score']) if pd.notna(r['severity_score']) else r['severity_score']}"
        )

summary = (
    "Fault segmentation summary for top-10 components (metrics):\n\n"
    f"1) Total fault runs found (point_count >=3): {count}\n"
    f"2) Earliest fault start timestamp: {earliest_ts}  ({earliest_dt_str})\n\n"
    "3) Notes:\n"
    "- Fault runs were derived by grouping consecutive anomaly minutes (exact 60s gaps) per (cmdb_id, kpi_name).\n"
    "- Severity score = max_breach_ratio * sqrt(duration_minutes). Many runs have p95==0 producing infinite breach ratios/severity.\n\n"
    "4) Example top fault runs (up to 5):\n"
    + ("\n".join(examples) if examples else "- No fault runs to show") + "\n\n"
    "Full fault-run table stored in kernel variable: metric_fault_runs_top\n"
    "If you want the CSV-like table of the top N runs, request metric_fault_runs_top.head(N)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(369, 1616635800,      cmdb_id                                           kpi_name  fault_start_timestamp  fault_end_timestamp  point_count  duration_minutes  max_value  p95  max_breach_ratio  earliest_anomaly_timestamp  severity_score
0   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             1616635800           1616637540           30                30        0.0  0.0               inf                  1616635800             inf
1   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             1616635800           1616637540           30                30        0.0  0.0               inf                  1616635800             inf
2   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             1616635800           1616637540           30                30        0.0  0.0               inf                  1616635800             inf
3   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             1616635800           1616637540           30                30        0.0  0.0               inf                  1616635800             inf
4   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             1616635800           1616637540           30                30        0.0  0.0               inf                  1616635800             inf
..       ...                                                ...                    ...                  ...          ...               ...        ...  ...               ...                         ...             ...
45   Mysql02                  Mysql-MySQL_3306_Com Update Multi             1616635860           1616637540           29                29        0.0  0.0               inf                  1616635860             inf
46   Mysql02          Mysql-MySQL_3306_CurrentSQLMaxRunningTime             1616635860           1616637540           29                29        1.0  0.0               inf                  1616635860             inf
47   Mysql02           Mysql-MySQL_3306_GetResponseTimeOfMysqld             1616635860           1616637540           29                29        0.0  0.0               inf                  1616635860             inf
48   Mysql02                 Mysql-MySQL_3306_Handler Savepoint             1616635860           1616637540           29                29        0.0  0.0               inf                  1616635860             inf
49   Mysql02        Mysql-MySQL_3306_Handler Savepoint Rollback             1616635860           1616637540           29                29        0.0  0.0               inf                  1616635860             inf

[50 rows x 11 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing variable metric_fault_runs_top from kernel
# Filter runs with finite p95 (p95 > 0), finite max_breach_ratio, and max_breach_ratio > 1.0

metric_fault_runs_finite = metric_fault_runs_top[
    (metric_fault_runs_top['p95'] > 0) &
    (np.isfinite(metric_fault_runs_top['max_breach_ratio'])) &
    (metric_fault_runs_top['max_breach_ratio'] > 1.0)
].copy()

# Aggregates
metric_finite_count = int(metric_fault_runs_finite.shape[0])

if metric_finite_count > 0:
    metric_finite_earliest = int(metric_fault_runs_finite['fault_start_timestamp'].min())
else:
    metric_finite_earliest = None

# Top 20 runs sorted by severity_score descending with requested columns
cols_out = [
    'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
    'point_count','max_value','p95','max_breach_ratio','severity_score'
]
metric_fault_runs_finite_top20 = metric_fault_runs_finite.sort_values('severity_score', ascending=False).reset_index(drop=True)[cols_out].head(20)

# Top 10 components by number of these finite-severity runs
top_components_finite = (
    metric_fault_runs_finite
    .groupby('cmdb_id', as_index=False)
    .size()
    .rename(columns={'size':'finite_fault_run_count'})
    .sort_values('finite_fault_run_count', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# Return compact outputs (variables)
metric_finite_count, metric_finite_earliest, metric_fault_runs_finite_top20, top_components_finite

```
Out[5]:
```
Summary of finite-severity metric fault runs (p95>0 and max_breach_ratio>1.0):

A) Total number of such runs: 1

B) Earliest fault_start_timestamp: 1616636400 (2021-03-25 09:40:00 Asia/Shanghai, UTC+8)

C) Top runs (up to 20) — columns: cmdb_id, kpi_name, fault_start, fault_end, point_count, max_value, p95, max_breach_ratio, severity_score:
- MG01 · OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — start=1616636400, end=1616636520, points=3, max_value=2.0, p95=1.0, max_breach_ratio=2.0, severity_score≈3.464102

Top components by count of these finite-severity runs (top 10):
- MG01 — finite_fault_run_count = 1

The filtered table is stored in kernel variable metric_fault_runs_finite for the next trace/log correlation step.

The original code execution output of IPython Kernel is also provided below for reference:

(1, 1616636400,   cmdb_id                                        kpi_name  fault_start_timestamp  fault_end_timestamp  point_count  max_value  p95  max_breach_ratio  severity_score
0    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             1616636400           1616636520            3        2.0  1.0               2.0        3.464102,   cmdb_id  finite_fault_run_count
0    MG01                       1)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: metric_fault_runs_finite, metric_anoms_window, trace_anoms_window, log_anoms_window, metric_thresholds
# Defensive copies (they are in-memory already)
mfr = metric_fault_runs_finite.copy() if 'metric_fault_runs_finite' in globals() else pd.DataFrame()
metric_anoms = metric_anoms_window.copy() if 'metric_anoms_window' in globals() else pd.DataFrame()
trace_anoms = trace_anoms_window.copy() if 'trace_anoms_window' in globals() else pd.DataFrame()
log_anoms = log_anoms_window.copy() if 'log_anoms_window' in globals() else pd.DataFrame()
metric_th = metric_thresholds.copy() if 'metric_thresholds' in globals() else pd.DataFrame()

# 1) metric_fault_runs_finite rows for MG01
mg01_fault_run = mfr[mfr['cmdb_id'] == 'MG01'].reset_index(drop=True)
# Limit to 20 for compact display
mg01_fault_run_display = mg01_fault_run.head(20)

# 2) trace anomalies for MG01 in incident window [1616635800, 1616639000]
start_ts = 1616635800
end_ts = 1616639000

mg01_trace_window = trace_anoms[
    (trace_anoms['cmdb_id'] == 'MG01') &
    (trace_anoms['timestamp'] >= start_ts) &
    (trace_anoms['timestamp'] <= end_ts)
].copy()

# (a) total number of trace anomaly rows for MG01
mg01_trace_total_rows = int(mg01_trace_window.shape[0])
# (b) number of distinct trace_name for MG01
mg01_trace_distinct_names = int(mg01_trace_window['trace_name'].nunique()) if not mg01_trace_window.empty else 0

# (c) top 20 trace series by anomaly_count with required columns
if not mg01_trace_window.empty:
    grp_trace = mg01_trace_window.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        anomaly_count = ('timestamp','count'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_value = ('value','max'),
        p95 = ('p95','first')
    )
    grp_trace['max_breach_ratio'] = np.where(grp_trace['p95'] == 0, np.inf, grp_trace['max_value'] / grp_trace['p95'])
    mg01_trace_top20 = grp_trace.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    mg01_trace_top20 = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95','max_breach_ratio'])

# 3) metric anomalies for MG01 with network-related KPI names (NETWORK or TCP or net, case-insensitive)
if not metric_anoms.empty:
    mask_net = metric_anoms['cmdb_id'].eq('MG01') & metric_anoms['kpi_name'].str.contains(r'NETWORK|TCP|net', case=False, na=False)
    mg01_metric_network_anoms = metric_anoms[mask_net].copy()
else:
    mg01_metric_network_anoms = pd.DataFrame(columns=metric_anoms.columns)

mg01_metric_network_total = int(mg01_metric_network_anoms.shape[0])
if not mg01_metric_network_anoms.empty:
    grp_metric_net = mg01_metric_network_anoms.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        anomaly_count = ('timestamp','count'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_value = ('value','max'),
        p95 = ('p95','first')
    )
    grp_metric_net['max_breach_ratio'] = np.where(grp_metric_net['p95'] == 0, np.inf, grp_metric_net['max_value'] / grp_metric_net['p95'])
    mg01_metric_net_top20 = grp_metric_net.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    mg01_metric_net_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95','max_breach_ratio'])

# 4) log anomalies for MG01 (if any)
if not log_anoms.empty:
    mg01_log_anoms = log_anoms[log_anoms['cmdb_id'] == 'MG01'].copy()
    if not mg01_log_anoms.empty:
        # Ensure required aggregated cols exist or compute per (log_name)
        # Provide requested columns per row with some appended aggregate cols for context
        # Compute per-log-name aggregates to include earliest_anomaly_timestamp, max_value, p95, max_breach_ratio
        agg_log = mg01_log_anoms.groupby(['cmdb_id','log_name'], as_index=False).agg(
            earliest_anomaly_timestamp = ('timestamp','min'),
            max_value = ('value','max'),
            p95 = ('p95','first')
        )
        agg_log['max_breach_ratio'] = np.where(agg_log['p95'] == 0, np.inf, agg_log['max_value'] / agg_log['p95'])
        # Join back the aggregates for display per-row (but limit rows to 20)
        mg01_log_anoms = mg01_log_anoms.merge(agg_log, on=['cmdb_id','log_name'], how='left')
        mg01_log_anoms_display = mg01_log_anoms[['timestamp','cmdb_id','log_name','value','earliest_anomaly_timestamp','max_value','p95','max_breach_ratio']].head(20).reset_index(drop=True)
    else:
        mg01_log_anoms_display = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value','earliest_anomaly_timestamp','max_value','p95','max_breach_ratio'])
else:
    mg01_log_anoms = pd.DataFrame()
    mg01_log_anoms_display = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value','earliest_anomaly_timestamp','max_value','p95','max_breach_ratio'])

# Store filtered results into kernel variables for follow-up
mg01_fault_run = mg01_fault_run  # full (but compact display below)
mg01_trace_anoms = mg01_trace_window
mg01_metric_network_anoms = mg01_metric_network_anoms
mg01_log_anoms = mg01_log_anoms

# Final compact outputs (limited to requested sizes)
mg01_fault_run_display, (mg01_trace_total_rows, mg01_trace_distinct_names, mg01_trace_top20), (mg01_metric_network_total, mg01_metric_net_top20), mg01_log_anoms_display

```
Out[6]:
```
Summary of MG01 TCP-CLOSE-WAIT fault corroboration (plain English):

1) Fault-run details (from metric_fault_runs_finite)
- One finite-severity fault run for MG01:
  - kpi_name: OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
  - start: 1616636400 (2021-03-25 09:40:00 +08)
  - end:   1616636520 (2021-03-25 09:42:00 +08)
  - point_count / duration_minutes: 3
  - max_value: 2.0, p95: 1.0, max_breach_ratio: 2.0
  - severity_score ≈ 3.464

2) Trace anomalies for MG01 (incident window 1616635800–1616639000)
- Total trace anomaly rows for MG01: 74
- Distinct trace_name values: 39
- Top trace series (examples from top 20):
  - trace.from_Tomcat01.row_count — anomaly_count=2, earliest=1616636640, max=440, p95=350, ratio≈1.257
  - trace.from_Tomcat04.row_count — anomaly_count=2, earliest=1616636160, max=431, p95=340, ratio≈1.268
  - trace.from_Tomcat03.row_count — anomaly_count=2, earliest=1616637300, max=459, p95=369, ratio≈1.244
  - trace.from_dockerB1.duration_mean — anomaly_count=2, earliest=1616636340, max≈0.184, p95≈0.127, ratio≈1.444 (one of the largest duration breach ratios)
  - Several other from_TomcatX and from_dockerX row_count/duration series show modest breaches (ratios ~1.03–1.27)
- Interpretation: multiple incoming trace series to MG01 (row counts and durations) show elevated counts or latencies around the same timeframe as the metric fault, indicating increased load and some latency increases from upstream services.

3) Network-related metric anomalies for MG01
- Total network-related anomaly rows (kpi_name contains NETWORK/TCP/net): 77
- Top network metric series (examples):
  - OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — anomaly_count=30, but p95==0 (ratio=inf) — many zero-baseline series produce infinite ratios and need careful interpretation.
  - OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — anomaly_count=3, earliest=1616636400, max=2.0, p95=1.0, ratio=2.0 (this is the finite-severity fault run above).
  - OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum — anomaly_count=2, max=6993, p95≈6267, ratio≈1.116
  - OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil and NETKBTotalPerSec show small breaches (ratios ≈1.01–1.04)
- Interpretation: the TCP-CLOSE-WAIT run is directly observed and other network KPIs show supporting small-to-moderate breaches (TotalTcpConnNum, bandwidth/KB/sec). Many network counters with p95==0 appear as inf ratios and should be interpreted with caution.

4) Log anomalies for MG01
- No log anomaly rows were found for MG01 in the window.

Overall conclusion
- The TCP-CLOSE-WAIT fault on MG01 at 09:40 (three consecutive minutes) is supported by:
  - A finite-severity metric fault run (p95>0, breach ratio 2.0).
  - Other network metrics showing additional breaches (TotalTcpConnNum, bandwidth/KB/sec).
  - Trace telemetry from multiple upstream components (Tomcat/docker sources) showing elevated row counts and some increased durations around the same timeframe.
- There are no MG01 log anomalies to confirm in logs.

Suggested next checks (operational, optional)
- Inspect MG01 socket/process state for CLOSE-WAIT sockets and identify the process holding sockets.
- Check connection_counts / accept queues and increase monitoring on TotalTcpConnNum and CLOSE-WAIT socket counts.
- Correlate with upstream services (Tomcat/docker hosts identified in traces) to see if they timed out/left sockets open.

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                        kpi_name  run_seq  fault_start_timestamp  fault_end_timestamp  point_count  max_value  p95  duration_minutes  earliest_anomaly_timestamp  max_breach_ratio  severity_score
0    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT        1             1616636400           1616636520            3        2.0  1.0                 3                  1616636400               2.0        3.464102, (74, 39,    cmdb_id                         trace_name  anomaly_count  earliest_anomaly_timestamp   max_value         p95  max_breach_ratio
0     MG01  trace.from_Tomcat01.duration_mean              2                  1616637000    0.534783    0.522631          1.023252
1     MG01   trace.from_Tomcat01.duration_p95              2                  1616636700    3.098300    3.089600          1.002816
2     MG01      trace.from_Tomcat01.row_count              2                  1616636640  440.000000  350.000000          1.257143
3     MG01  trace.from_Tomcat02.duration_mean              2                  1616637060    0.555177    0.537482          1.032921
4     MG01   trace.from_Tomcat02.duration_p95              2                  1616636160    3.260400    3.152825          1.034120
5     MG01      trace.from_Tomcat02.row_count              2                  1616636640  430.000000  368.000000          1.168478
6     MG01      trace.from_Tomcat03.row_count              2                  1616637300  459.000000  369.000000          1.243902
7     MG01        trace.to_dockerB1.row_count              2                  1616636640  714.000000  644.500000          1.107836
8     MG01  trace.from_Tomcat04.duration_mean              2                  1616637000    0.548900    0.532985          1.029860
9     MG01   trace.from_Tomcat04.duration_p95              2                  1616636640    3.251300    3.101250          1.048384
10    MG01      trace.from_Tomcat04.row_count              2                  1616636160  431.000000  340.000000          1.267647
11    MG01   trace.from_dockerA1.duration_p95              2                  1616636340    1.044150    1.034060          1.009758
12    MG01  trace.from_dockerA1.duration_mean              2                  1616636460    0.208367    0.165774          1.256932
13    MG01      trace.from_dockerA1.row_count              2                  1616636340   85.000000   73.650000          1.154107
14    MG01  trace.from_dockerA2.duration_mean              2                  1616636400    0.195769    0.164924          1.187026
15    MG01   trace.from_dockerB1.duration_p95              2                  1616636340    1.035000    1.024950          1.009805
16    MG01   trace.from_dockerA2.duration_p95              2                  1616636400    1.035700    1.031850          1.003731
17    MG01      trace.from_dockerA2.row_count              2                  1616636580   78.000000   70.500000          1.106383
18    MG01  trace.from_dockerB1.duration_mean              2                  1616636340    0.184082    0.127441          1.444443
19    MG01  trace.from_dockerB2.duration_mean              2                  1616636400    0.178830    0.152297          1.174223), (77,   cmdb_id                                         kpi_name  anomaly_count  earliest_anomaly_timestamp    max_value           p95  max_breach_ratio
0    MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             30                  1616635800        0.000  0.000000e+00               inf
1    MG01     OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc             10                  1616635800        0.000  0.000000e+00               inf
2    MG01          OSLinux-OSLinux_NETWORK_ens160_NETInErr             10                  1616635800        0.000  0.000000e+00               inf
3    MG01         OSLinux-OSLinux_NETWORK_ens160_NETOutErr             10                  1616635800        0.000  0.000000e+00               inf
4    MG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc             10                  1616635800        0.000  0.000000e+00               inf
5    MG01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT              3                  1616636400        2.000  1.000000e+00          2.000000
6    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum              2                  1616637300     6993.000  6.267000e+03          1.115845
7    MG01  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil              1                  1616637000        0.011  1.052000e-02          1.045627
8    MG01  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec              1                  1616636760  1023272.000  1.006746e+06          1.016416), Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value, earliest_anomaly_timestamp, max_value, p95, max_breach_ratio]
Index: [])```
```